In [ ]:
# Info do catalog do desenvolvedor
catalog_dev = dbutils.widgets.get("catalog_dev")
schema = dbutils.widgets.get("schema")

In [ ]:
lista_dir = ["vendas", "devolucoes", "movimentacao_estoque"]

for dir in lista_dir:
    caminho_volume = f"/Volumes/{catalog_dev}/{schema}/volume_arquivos_raw/{dir}"
    caminho_checkpoint = f"/Volumes/{catalog_dev}/{schema}/volume_arquivos_raw/_checkpoints/{dir}/"
    caminho_tabelas = f"{catalog_dev}/{schema}/"

    # Leitura dos arquivos CSV no volume
    df_bronze_csv = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("pathGlobFilter", "*.csv")
        .option("header", "true")
        .option("cloudFiles.includeExistingFiles", "true")
        .option("cloudFiles.schemaLocation", caminho_tabelas)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes", "true") # Deixa o Autoloader inferir os tipos brutos
        .option("cloudFiles.schemaLocation", caminho_checkpoint)
        .option("cloudFiles.maxFilesPerTrigger", "3")
        .load(caminho_volume)
    )

    # Escrita na tabela Bronze exclusiva dos CSVs
    query = (
        df_bronze_csv
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("mergeSchema", "true")
        .option("checkpointLocation", caminho_checkpoint)
        .trigger(availableNow=True)
        .toTable(f"{catalog_dev}.{schema}.{dir}_stream")
    )